### FIGURES FOR UNCERTAINTY-AWARE REINFORCEMENT LEARNING FOR CHEMICAL LANGUAGE MODELS

Some of the figures take a lot of memory and time to plot, they are composed by a lot of long tables and therefore extremely expensive in terms of computing

In [17]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import Chem

from glob import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import os
from scipy.ndimage import gaussian_filter1d

# ── Publication-quality rcParams ──────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "savefig.facecolor": "white",
    "font.family":       "DejaVu Sans",
    "font.size":         9,
    "axes.titlesize":    10,
    "axes.labelsize":    9,
    "xtick.labelsize":   8,
    "ytick.labelsize":   8,
    "legend.fontsize":   9,
    "axes.linewidth":    0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size":  3.0,
    "ytick.major.size":  3.0,
    "xtick.minor.visible": False,
    "ytick.minor.visible": False,
    "lines.linewidth":   1.2,
    "patch.linewidth":   0.0,
    "pdf.fonttype":      42,   # embed fonts properly
    "ps.fonttype":       42,
})


### ANALYSIS OF MODEL SYSTEM

#### Unique noisy component

In [ ]:


# Colorblind-safe palette (Wong 2011)
LABELS = [
    "No Noise",
    "Noisy Component",
    "Noisy Component + Loss Modulation",
    "Noisy Component + Score Modulation",
    "Noisy Component + Score & Loss Modulation",
]


COLORS = [
    "#555555",
    "#70C267",
    "#6D85BE",  
    "#E66532",
    "#9B2F70" 
]


mid = [
    "./02_RL/uniqueNoise_mid/DISTw0_SFw1_N5NoNoise_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw0_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw0_SFw1_N5EucReward_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw1_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw1_SFw1_N5EucReward_1*.csv",
]

low = [
    "./02_RL/uniqueNoise_low/DISTw0_SFw1_N5NoNoise_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw0_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw0_SFw1_N5EucReward_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw1_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw1_SFw1_N5EucReward_1*.csv",
]

# Figure layout 
PANEL_LABELS = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]
TITLES       = ["Acc. Predicted Hits", "Accumulated False Hits", "True Hit Ratio", "Final Score", "Predicted LogP", "Uncertainty | Distances"]
Y_LABELS     = ["Acc. Predicted Hits", "Accumulated false hits", "True / total hits", "Mean final score", "Mean predicted LogP", "Mean transformed distance"]

def fmt_kilo(x, _):
    return f"{int(x/1000)}k" if x >= 1000 else ("0" if x == 0 else f"{int(x)}")
def to_run_array(series_list, full_index):
    return np.array([
        s.reindex(full_index).ffill().fillna(0).to_numpy() for s in series_list
    ])

fig, axes = plt.subplots(2, 3, figsize=(9, 5), dpi=600, constrained_layout=True, sharex=True)

for i, ax in enumerate(axes.flatten()):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(direction="out", which="both", length=3, width=0.8)
    ax.grid(True, alpha=0.18, linestyle=":", linewidth=0.6, color="#888888")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_kilo))
    ax.set_title(TITLES[i], fontweight="bold", pad=20)
    ax.set_xlabel("RL step")
    ax.set_ylabel(Y_LABELS[i])
    # Panel label (a), (b), (c) in upper-left corner
    ax.text(-0.15, 1.3, PANEL_LABELS[i], transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top", ha="left")

#  Data processing & plotting (logic unchanged) 
for file, col in zip(low, COLORS):
    files = glob(file)[:5]  # Limit to first 5 runs for consistency
    print(files)

    accumulated_hits_list      = []
    accumulated_true_hits_list = []
    exist = False

    for exp in files:
        if os.path.exists(exp):
            exist = True
            
            df = pd.read_csv(exp)
            df = df.dropna(subset=["Scaffold"])
            df = df.round(4).dropna()
            
            if df["step"].max() > 7000:
                print(df["step"].max())
                mask = (
                    (df["step"] < 10000) &
                    (df["UncertaintyLogP (raw)"] > 2.05) &
                    (df["UncertaintyLogP (raw)"] < 2.95) &
                    (df["MW"] > 0.65) &
                    (df["TPSA"] > 0.65)
                )
                hits        = df[mask].copy()
                unique_hits = hits.drop_duplicates(subset="Scaffold").reset_index(drop=True)
                true_hits   = unique_hits[
                    (unique_hits["logp_actual (UncertaintyLogP)"] > 2.05) &
                    (unique_hits["logp_actual (UncertaintyLogP)"] < 2.95)
                ]

                hitsList=unique_hits["step"].value_counts().sort_index().cumsum()
                hitsTrueList=true_hits["step"].value_counts().sort_index().cumsum()

                accumulated_hits_list.append(hitsList)
                accumulated_true_hits_list.append(hitsTrueList)
                #print(max(df["step"]),hitsList.max(),hitsTrueList.max())
    if exist:
        full_index = np.arange(10000)
        reindex = lambda lst: np.array([
            s.reindex(full_index).ffill().fillna(0).to_numpy() for s in lst
        ])
        hits_arr  = reindex(accumulated_hits_list)
        true_arr  = reindex(accumulated_true_hits_list)
        false_arr = hits_arr - true_arr

        mean_hits = hits_arr.mean(0)
        mean_true = true_arr.mean(0)
        mean_false = false_arr.mean(0)
        std_hits  = hits_arr.std(0)
        std_true  = true_arr.std(0)
        std_false  = false_arr.std(0)

        kw_fill = dict(alpha=0.18, color=col, linewidth=0)
        kw_line = dict(linewidth=1.2, color=col)

        axes[0,0].plot(full_index, mean_hits, **kw_line)
        axes[0,0].fill_between(full_index, mean_hits - std_hits, mean_hits + std_hits, **kw_fill)

        axes[0,1].plot(full_index, mean_false, **kw_line)
        axes[0,1].fill_between(full_index, mean_false - std_false, mean_false + std_false, **kw_fill)

        if "NoNoise"  not in files[0]:
            # Guard against division by zero in ratio panel
            denom      = np.where(mean_hits[10:] > 0, mean_hits[10:], np.nan)
            ratio_mean = mean_true[10:] / denom
            axes[0,2].plot(full_index[10:], ratio_mean, **kw_line)
            axes[0,2].set_ylim(0.2, 1)

for pattern, col in zip(low, COLORS):
    files = sorted(glob(pattern))[:5]  # Limit to first 5 runs for consistency
    score_list, logp_list, sim_list = [], [], []

    for exp in files:
        if not os.path.exists(exp):
            continue

        df = pd.read_csv(exp)
        df = df.dropna(subset=["Scaffold"]).round(4).dropna()
        df = df[df["step"] < 10000].copy()
        if df["step"].max() <= 9000:
            continue
        

        cols = ["Score", "UncertaintyLogP", "DistanceEGFR"]
        step_means = df.groupby("step")[cols].mean()

        score_list.append(step_means["Score"])
        logp_list.append(step_means["UncertaintyLogP"])
        sim_list.append(1-step_means["DistanceEGFR"])

    if not score_list:
        continue

    full_index = np.arange(10000)
    score_arr = to_run_array(score_list, full_index)
    logp_arr = to_run_array(logp_list, full_index)
    sim_arr = to_run_array(sim_list, full_index)

    metric_arrays = [score_arr, logp_arr, sim_arr]

    for ax_idx, arr in enumerate(metric_arrays):
        mean_v = arr.mean(axis=0)
        std_v = arr.std(axis=0)

        mean_s = gaussian_filter1d(mean_v, sigma=8)

        axes[1,ax_idx].plot(full_index, mean_s, color=col, linewidth=1)
        axes[1,ax_idx].fill_between(
            full_index,
            mean_s - std_v,
            mean_s + std_v,
            color=col,
            alpha=0.18,
            linewidth=0,
        )
        axes[1,ax_idx].set_ylim(0, 1)



# Legend 
handles = [
    Line2D([0], [0], color=col, lw=1.6, label=lbl, solid_capstyle="round")
    for lbl, col in zip(LABELS, COLORS)
]
blank = Line2D([0], [0], color="none", label="")
# Insert blank at index 2 so row2 reads: [blank, item3, item4] → centered
handles_padded = handles[:1] + [blank] + handles[1:]
fig.legend(
    handles=handles_padded,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.1),
    ncol=3,
    frameon=False,
    handlelength=1.6,
    handletextpad=0.5,
    columnspacing=1.2,
    labelspacing=0.35,
)

plt.savefig("03_results/accHits_low.pdf", dpi=300, bbox_inches="tight")
plt.savefig("03_results/accHits_low.png", dpi=300, bbox_inches="tight")
plt.show()

Model system Small

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import Chem

from glob import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import os
from scipy.ndimage import gaussian_filter1d


# Colorblind-safe palette 
LABELS = [
    "No Noise",
    "Noisy Component",
    "Noisy Component + Loss Modulation",
    "Noisy Component + Score Modulation",
    "Noisy Component + Score & Loss Modulation",
]


COLORS = [
    "#555555",
    "#70C267",
    "#6D85BE",  
    "#E66532",
    "#9B2F70" 
]


mid = [
    "./02_RL/uniqueNoise_mid/DISTw0_SFw1_N5NoNoise_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw0_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw0_SFw1_N5EucReward_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw1_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw1_SFw1_N5EucReward_1*.csv",
]

low = [
    "./02_RL/uniqueNoise_low/DISTw0_SFw1_N5NoNoise_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw0_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw0_SFw1_N5EucReward_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw1_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw1_SFw1_N5EucReward_1*.csv",
]


#  Figure layout
PANEL_LABELS = ["(a)", "(b)", "(c)"]
TITLES       = ["Acc. Predicted Hits", "True Hit Ratio", "Uncertainty | Distances"]
Y_LABELS     = ["Acc. Predicted Hits", "True / total hits", "Mean transformed distance"]

def fmt_kilo(x, _):
    return f"{int(x/1000)}k" if x >= 1000 else ("0" if x == 0 else f"{int(x)}")
def to_run_array(series_list, full_index):
    return np.array([
        s.reindex(full_index).ffill().fillna(0).to_numpy() for s in series_list
    ])

fig, axes = plt.subplots(1, 3, figsize=(9, 3), dpi=600, constrained_layout=True, sharex=True)

for i, ax in enumerate(axes.flatten()):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(direction="out", which="both", length=3, width=0.8)
    ax.grid(True, alpha=0.18, linestyle=":", linewidth=0.6, color="#888888")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_kilo))
    ax.set_title(TITLES[i], fontweight="bold", pad=20)
    ax.set_xlabel("RL step")
    ax.set_ylabel(Y_LABELS[i])
    # Panel label (a), (b), (c) in upper-left corner
    ax.text(-0.15, 1.4, PANEL_LABELS[i], transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top", ha="left")

#Data processing & plotting (logic unchanged) 
for file, col in zip(mid, COLORS):
    files = glob(file)
    print(files)

    accumulated_hits_list      = []
    accumulated_true_hits_list = []
    exist = False

    for exp in files:
        if os.path.exists(exp):
            exist = True
            df = pd.read_csv(exp)
            df = df.dropna(subset=["Scaffold"])
            df = df.round(4).dropna()
            if df["step"].max() > 8000:
                print(df["step"].max())
                mask = (
                (df["step"]<10000) &
                (df["UncertaintyLogP (raw)"] > 2.05) &
                (df["UncertaintyLogP (raw)"] < 2.95) &
                (df["MW"] > 0.65) &
                (df["TPSA"] > 0.65)
                )
                hits        = df[mask].copy()
                unique_hits = hits.drop_duplicates(subset="Scaffold").reset_index(drop=True)
                true_hits   = unique_hits[
                    (unique_hits["logp_actual (UncertaintyLogP)"] > 2.05) &
                    (unique_hits["logp_actual (UncertaintyLogP)"] < 2.95) 
                ]
                accumulated_hits_list.append(unique_hits["step"].value_counts().sort_index().cumsum())
                accumulated_true_hits_list.append(true_hits["step"].value_counts().sort_index().cumsum())

    if exist:
        full_index = np.arange(10000)
        reindex = lambda lst: np.array([
            s.reindex(full_index).ffill().fillna(0).to_numpy() for s in lst
        ])
        hits_arr  = reindex(accumulated_hits_list)
        true_arr  = reindex(accumulated_true_hits_list)
        false_arr = hits_arr - true_arr

        mean_hits = hits_arr.mean(0)
        mean_true = true_arr.mean(0)
        mean_false = false_arr.mean(0)
        std_hits  = hits_arr.std(0)
        std_true  = true_arr.std(0)
        std_false  = false_arr.std(0)

        kw_fill = dict(alpha=0.18, color=col, linewidth=0)
        kw_line = dict(linewidth=1.2, color=col)

        axes[0].plot(full_index, mean_hits, **kw_line)
        axes[0].fill_between(full_index, mean_hits - std_hits, mean_hits + std_hits, **kw_fill)

        if "NoNoise"  not in files[0]:
            denom      = np.where(mean_hits[100:] > 0, mean_hits[100:], np.nan)
            ratio_mean = mean_true[100:] / denom
            axes[1].plot(full_index[100:], ratio_mean, **kw_line)
            axes[1].set_ylim(0.2, 1)

for pattern, col in zip(mid, COLORS):
    files = sorted(glob(pattern))
    score_list, logp_list, bertz_list, simEGFR_list, simDRD2_list = [], [], [], [], []

    for exp in files:
        if not os.path.exists(exp):
            continue

        df = pd.read_csv(exp)
        df = df.dropna(subset=["Scaffold"]).round(4).dropna()
        df = df[df["step"] < 10000].copy()
        if df["step"].max() <= 8000:
            continue

        cols = ["DistanceEGFR"]
        step_means = df.groupby("step")[cols].mean()

        simEGFR_list.append(1-step_means["DistanceEGFR"])


    full_index = np.arange(10000)
    simEGFR_arr = to_run_array(simEGFR_list, full_index)



    #scoresConj = (logp_arr + bertz_arr)/2
    #simsConj = (simEGFR_arr + simDRD2_arr)/2
    mean_v = simEGFR_arr.mean(axis=0)
    std_v = simEGFR_arr.std(axis=0)

    mean_s = gaussian_filter1d(mean_v, sigma=8)
    axes[2].plot(full_index, mean_s, color=col, linewidth=1)
    axes[2].fill_between(
            full_index,
            mean_s - std_v,
            mean_s + std_v,
            color=col,
            alpha=0.18,
            linewidth=0,
        )
    axes[2].set_ylim(0, 1)



# Legend 
handles = [
    Line2D([0], [0], color=col, lw=1.6, label=lbl, solid_capstyle="round")
    for lbl, col in zip(LABELS, COLORS)
]
blank = Line2D([0], [0], color="none", label="")
# Insert blank at index 2 so row2 reads: [blank, item3, item4] → centered
handles_padded = handles[:1] + [blank] + handles[1:]
fig.legend(
    handles=handles_padded,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.2),
    ncol=3,
    frameon=False,
    handlelength=1.6,
    handletextpad=0.5,
    columnspacing=1.2,
    labelspacing=0.35,
)

plt.tight_layout()
plt.savefig("03_results/accHits_Mid_Unique_midS.pdf", dpi=300, bbox_inches="tight")
plt.savefig("03_results/accHits_Mid_Unique_midS.png", dpi=300, bbox_inches="tight")
plt.show()
plt.show()

Distance Distributions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from glob import glob
import pandas as pd
import numpy as np

COLORS = [
    "#555555",
    "#70C267",
    "#6D85BE",  
    "#E66532",
    "#9B2F70" 
]
LABELS = [
    "No Noise",
    "Noisy Component",
    "Noisy Component + Loss Modulation",
    "Noisy Component + Score Modulation",
    "Noisy Component + Score & Loss Modulation",
]


mid = [
    "./02_RL/uniqueNoise_mid/DISTw0_SFw1_N5NoNoise_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw0_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw0_SFw1_N5EucReward_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw1_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw1_SFw1_N5EucReward_1*.csv",
]

low = [
    "./02_RL/uniqueNoise_low/DISTw0_SFw1_N5NoNoise_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw0_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw0_SFw1_N5EucReward_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw1_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw1_SFw1_N5EucReward_1*.csv",
]


def plot_densities(ax,files, colors, labels):

    for pattern, color, label in zip(files, colors, labels):
        files = glob(pattern)
        all_distances = []

        for f in files:
                df_tmp = pd.read_csv(f)
                df_tmp = df_tmp[df_tmp["step"] < 200].copy()
                vals = df_tmp["DistanceEGFR"].dropna().to_numpy()
                if len(vals) > 1:
                    all_distances.append(1-vals)

        if not all_distances:
                continue

        # Common x-grid for this condition
        x_min = min(arr.min() for arr in all_distances)
        x_max = max(arr.max() for arr in all_distances)
        x_grid = np.linspace(x_min, x_max, 400)

        # KDE per file, then mean KDE
        densities = np.array([gaussian_kde(arr)(x_grid) for arr in all_distances])
        mean_density = densities.mean(axis=0)

        ax.plot(x_grid, mean_density, color=color, linewidth=2, label=label)

    ax.set_xlabel("Distance EGFR")
    ax.set_ylabel("Mean density")

fig,axes= plt.subplots(1, 2, figsize=(10, 5), dpi=600, sharey=True)

plot_densities(axes[0], mid, COLORS, LABELS)
plot_densities(axes[1], low, COLORS, LABELS)

axes[0].set_title("Unique Noise - Mid-initial-distances", fontweight="bold")
axes[1].set_title("Unique Noise - High-initial-distances", fontweight="bold")

plt.tight_layout()

# Legend 
handles = [
    Line2D([0], [0], color=col, lw=1.6, label=lbl, solid_capstyle="round")
    for lbl, col in zip(LABELS, COLORS)
]
blank = Line2D([0], [0], color="none", label="")
# Insert blank at index 2 so row2 reads: [blank, item3, item4] → centered
handles_padded = handles[:1] + [blank] + handles[1:]
fig.legend(
    handles=handles_padded,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.1),
    ncol=3,
    frameon=False,
    handlelength=1.6,
    handletextpad=0.5,
    columnspacing=1.2,
    labelspacing=0.35,
)

plt.tight_layout()
#plt.savefig("03_results/distance_distribution_mean_kde.pdf", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from glob import glob
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D

COLORS = [
    "#555555",
    "#70C267",
    "#6D85BE",  
    "#E66532",
    "#9B2F70" 
]
LABELS = [
    "No Noise",
    "Noisy Component",
    "Noisy Component + Loss Modulation",
    "Noisy Component + Score Modulation",
    "Noisy Component + Score & Loss Modulation",
]

mid = [
    "./02_RL/uniqueNoise_mid/DISTw0_SFw1_N5NoNoise_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw0_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw0_SFw1_N5EucReward_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw1_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_mid/DISTw1_SFw1_N5EucReward_1*.csv",
]

low = [
    "./02_RL/uniqueNoise_low/DISTw0_SFw1_N5NoNoise_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw0_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw0_SFw1_N5EucReward_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw1_SFw1_N5Euc_1*.csv",
    "./02_RL/uniqueNoise_low/DISTw1_SFw1_N5EucReward_1*.csv",
]


def plot_densities(ax, patterns, colors, labels, max_step=None):
    for pattern, color, label in zip(patterns, colors, labels):
        matched_files = glob(pattern)
        all_distances = []

        for f in matched_files:
            df_tmp = pd.read_csv(f)
            if max_step is not None:
                df_tmp = df_tmp[df_tmp["step"] < max_step].copy()
            vals = df_tmp["DistanceEGFR"].dropna().to_numpy()
            if len(vals) > 1:
                all_distances.append(1 - vals)

        if not all_distances:
            continue

        x_min = min(arr.min() for arr in all_distances)
        x_max = max(arr.max() for arr in all_distances)
        x_grid = np.linspace(x_min, x_max, 400)

        densities = np.array([gaussian_kde(arr)(x_grid) for arr in all_distances])
        mean_density = densities.mean(axis=0)

        ax.plot(x_grid, mean_density, color=color, linewidth=2, label=label)

    ax.set_xlabel("Similarity EGFR")
    ax.set_ylabel("Mean density")


fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=600, sharey="row")

# Top row: first 100 steps
plot_densities(axes[0, 0], mid, COLORS, LABELS, max_step=100)
plot_densities(axes[0, 1], low, COLORS, LABELS, max_step=100)
axes[0, 0].set_title("Unique Noise - Mid (steps 0–100)", fontweight="bold")
axes[0, 1].set_title("Unique Noise - High (steps 0–100)", fontweight="bold")

# Bottom row: full run (10000 steps)
plot_densities(axes[1, 0], mid, COLORS, LABELS, max_step=10000)
plot_densities(axes[1, 1], low, COLORS, LABELS, max_step=10000)
axes[1, 0].set_title("Unique Noise - Mid (full run)", fontweight="bold")
axes[1, 1].set_title("Unique Noise - High (full run)", fontweight="bold")

for ax in axes.flatten():
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()

# Legend
handles = [
    Line2D([0], [0], color=col, lw=1.6, label=lbl, solid_capstyle="round")
    for lbl, col in zip(LABELS, COLORS)
]
blank = Line2D([0], [0], color="none", label="")
handles_padded = handles[:1] + [blank] + handles[1:]
fig.legend(
    handles=handles_padded,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=3,
    frameon=False,
    handlelength=1.6,
    handletextpad=0.5,
    columnspacing=1.2,
    labelspacing=0.35,
)

plt.tight_layout()
plt.savefig("03_results/distance_distribution_All.pdf", dpi=600, bbox_inches="tight")
plt.show()

#### Multiple noisy component

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import Chem

from glob import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import os
from scipy.ndimage import gaussian_filter1d


# Colorblind-safe palette 
LABELS = [
    "No Noise",
    "Noisy Component",
    "Noisy Component + Loss Modulation",
    "Noisy Component + Score Modulation",
    "Noisy Component + Score & Loss Modulation",
]


COLORS = [
    "#555555",
    "#70C267",
    "#6D85BE",  
    "#E66532",
    "#9B2F70" 
]



mid = [
    "./02_RL/multipleNoise/DISTw0_SFw1_N5NoNoise_1*.csv",
    "./02_RL/multipleNoise/DISTw0_SFw1_N5Euc_1*.csv",
    "./02_RL/multipleNoise/DISTw0_SFw1_N5EucReward_1*.csv",
    "./02_RL/multipleNoise/DISTw1_SFw1_N5Euc_1*.csv",
    "./02_RL/multipleNoise/DISTw1_SFw1_N5EucReward_1*.csv",
]

low = [
    "./02_RL/multipleNoise_low/DISTw0_SFw1_N5NoNoise_1*.csv",
    "./02_RL/multipleNoise_low/DISTw0_SFw1_N5Euc_1*.csv",
    "./02_RL/multipleNoise_low/DISTw0_SFw1_N5EucReward_1*.csv",
    "./02_RL/multipleNoise_low/DISTw1_SFw1_N5Euc_1*.csv",
    "./02_RL/multipleNoise_low/DISTw1_SFw1_N5EucReward_1*.csv",
]

#  Figure layout
PANEL_LABELS = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]
TITLES       = ["Acc. Predicted Hits", "Accumulated False Hits", "True Hit Ratio", "Final Score", "Predicted LogP&BertzCT", "Uncertainty | Distances"]
Y_LABELS     = ["Acc. Predicted Hits", "Accumulated false hits", "True / total hits", "Mean final score", "Mean LogP&BertzCT", "Mean transformed distance"]

def fmt_kilo(x, _):
    return f"{int(x/1000)}k" if x >= 1000 else ("0" if x == 0 else f"{int(x)}")
def to_run_array(series_list, full_index):
    return np.array([
        s.reindex(full_index).ffill().fillna(0).to_numpy() for s in series_list
    ])

fig, axes = plt.subplots(2, 3, figsize=(9, 5), dpi=600, constrained_layout=True, sharex=True)

for i, ax in enumerate(axes.flatten()):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(direction="out", which="both", length=3, width=0.8)
    ax.grid(True, alpha=0.18, linestyle=":", linewidth=0.6, color="#888888")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_kilo))
    ax.set_title(TITLES[i], fontweight="bold", pad=20)
    ax.set_xlabel("RL step")
    ax.set_ylabel(Y_LABELS[i])
    # Panel label (a), (b), (c) in upper-left corner
    ax.text(-0.15, 1.4, PANEL_LABELS[i], transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top", ha="left")

#Data processing & plotting (logic unchanged) 
for file, col in zip(low, COLORS):
    files = glob(file)
    print(files)

    accumulated_hits_list      = []
    accumulated_true_hits_list = []
    exist = False

    for exp in files:
        if os.path.exists(exp):
            exist = True
            df = pd.read_csv(exp)
            df = df.dropna(subset=["Scaffold"])
            df = df.round(4).dropna()
            if df["step"].max() > 8000:
                print(df["step"].max())
                mask = (
                (df["step"]<10000) &
                (df["UncertaintyLogP (raw)"] > 2.05) &
                (df["UncertaintyLogP (raw)"] < 2.95) &
                (df["UncertaintyBertz (raw)"] > 810) &
                (df["UncertaintyBertz (raw)"] < 990) &
                (df["MW"] > 0.65) &
                (df["TPSA"] > 0.65)
                )
                hits        = df[mask].copy()
                unique_hits = hits.drop_duplicates(subset="Scaffold").reset_index(drop=True)
                true_hits   = unique_hits[
                    (unique_hits["logp_actual (UncertaintyLogP)"] > 2.05) &
                    (unique_hits["logp_actual (UncertaintyLogP)"] < 2.95) &
                    (unique_hits["Bertz_actual (UncertaintyBertz)"] > 810) &
                    (unique_hits["Bertz_actual (UncertaintyBertz)"] < 990)
                ]
                accumulated_hits_list.append(unique_hits["step"].value_counts().sort_index().cumsum())
                accumulated_true_hits_list.append(true_hits["step"].value_counts().sort_index().cumsum())

    if exist:
        full_index = np.arange(10000)
        reindex = lambda lst: np.array([
            s.reindex(full_index).ffill().fillna(0).to_numpy() for s in lst
        ])
        hits_arr  = reindex(accumulated_hits_list)
        true_arr  = reindex(accumulated_true_hits_list)
        false_arr = hits_arr - true_arr

        mean_hits = hits_arr.mean(0)
        mean_true = true_arr.mean(0)
        mean_false = false_arr.mean(0)
        std_hits  = hits_arr.std(0)
        std_true  = true_arr.std(0)
        std_false  = false_arr.std(0)

        kw_fill = dict(alpha=0.18, color=col, linewidth=0)
        kw_line = dict(linewidth=1.2, color=col)

        axes[0,0].plot(full_index, mean_hits, **kw_line)
        axes[0,0].fill_between(full_index, mean_hits - std_hits, mean_hits + std_hits, **kw_fill)

        axes[0,1].plot(full_index, mean_false, **kw_line)
        axes[0,1].fill_between(full_index, mean_false - std_false, mean_false + std_false, **kw_fill)
        
        if "NoNoise"  not in files[0]:
            # Guard against division by zero in ratio panel
            denom      = np.where(mean_hits[100:] > 0, mean_hits[100:], np.nan)
            ratio_mean = mean_true[100:] / denom
            axes[0,2].plot(full_index[100:], ratio_mean, **kw_line)
            axes[0,2].set_ylim(0.2, 1)

for pattern, col in zip(low, COLORS):
    files = sorted(glob(pattern))
    score_list, logp_list, bertz_list, simEGFR_list, simDRD2_list = [], [], [], [], []

    for exp in files:
        if not os.path.exists(exp):
            continue

        df = pd.read_csv(exp)
        df = df.dropna(subset=["Scaffold"]).round(4).dropna()
        df = df[df["step"] < 10000].copy()
        if df["step"].max() <= 8000:
            continue

        cols = ["Score", "UncertaintyLogP", "UncertaintyBertz", "DistanceEGFR", "DistanceDRD2"]
        step_means = df.groupby("step")[cols].mean()

        score_list.append(step_means["Score"])
        logp_list.append(step_means["UncertaintyLogP"])
        bertz_list.append(step_means["UncertaintyBertz"])
        simEGFR_list.append(1-step_means["DistanceEGFR"])
        simDRD2_list.append(1-step_means["DistanceDRD2"])

    if not score_list:
        continue

    full_index = np.arange(10000)
    score_arr = to_run_array(score_list, full_index)
    logp_arr = to_run_array(logp_list, full_index)
    bertz_arr = to_run_array(bertz_list, full_index)
    simEGFR_arr = to_run_array(simEGFR_list, full_index)
    simDRD2_arr = to_run_array(simDRD2_list, full_index)

    scoresConj = np.sqrt(logp_arr * bertz_arr)
    simsConj = np.sqrt(simEGFR_arr * simDRD2_arr)
    #scoresConj = (logp_arr + bertz_arr)/2
    #simsConj = (simEGFR_arr + simDRD2_arr)/2

    metric_arrays = [score_arr, scoresConj, simsConj]


    for ax_idx, arr in enumerate(metric_arrays):
        mean_v = arr.mean(axis=0)
        std_v = arr.std(axis=0)

        mean_s = gaussian_filter1d(mean_v, sigma=8)

        axes[1,ax_idx].plot(full_index, mean_s, color=col, linewidth=1)
        axes[1,ax_idx].fill_between(
            full_index,
            mean_s - std_v,
            mean_s + std_v,
            color=col,
            alpha=0.18,
            linewidth=0,
        )
        axes[1,ax_idx].set_ylim(0, 1)



# Legend 
handles = [
    Line2D([0], [0], color=col, lw=1.6, label=lbl, solid_capstyle="round")
    for lbl, col in zip(LABELS, COLORS)
]
blank = Line2D([0], [0], color="none", label="")
# Insert blank at index 2 so row2 reads: [blank, item3, item4] → centered
handles_padded = handles[:1] + [blank] + handles[1:]
fig.legend(
    handles=handles_padded,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.1),
    ncol=3,
    frameon=False,
    handlelength=1.6,
    handletextpad=0.5,
    columnspacing=1.2,
    labelspacing=0.35,
)

plt.tight_layout()
plt.savefig("03_results/accHits_Mid_Multi_low.pdf", dpi=300, bbox_inches="tight")
plt.savefig("03_results/accHits_Mid_Multi_low.png", dpi=300, bbox_inches="tight")
plt.show()

Small version of the plot for the main text

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import Chem

from glob import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import os
from scipy.ndimage import gaussian_filter1d


# Colorblind-safe palette 
LABELS = [
    "No Noise",
    "Noisy Component",
    "Noisy Component + Loss Modulation",
    "Noisy Component + Score Modulation",
    "Noisy Component + Score & Loss Modulation",
]


COLORS = [
    "#555555",
    "#70C267",
    "#6D85BE",  
    "#E66532",
    "#9B2F70" 
]


mid = [
    "./02_RL/multipleNoise/DISTw0_SFw1_N5NoNoise_1*.csv",
    "./02_RL/multipleNoise/DISTw0_SFw1_N5Euc_1*.csv",
    "./02_RL/multipleNoise/DISTw0_SFw1_N5EucReward_1*.csv",
    "./02_RL/multipleNoise/DISTw1_SFw1_N5Euc_1*.csv",
    "./02_RL/multipleNoise/DISTw1_SFw1_N5EucReward_1*.csv",
]

low = [
    "./02_RL/multipleNoise_low/DISTw0_SFw1_N5NoNoise_1*.csv",
    "./02_RL/multipleNoise_low/DISTw0_SFw1_N5Euc_1*.csv",
    "./02_RL/multipleNoise_low/DISTw0_SFw1_N5EucReward_1*.csv",
    "./02_RL/multipleNoise_low/DISTw1_SFw1_N5Euc_1*.csv",
    "./02_RL/multipleNoise_low/DISTw1_SFw1_N5EucReward_1*.csv",
]

#  Figure layout
PANEL_LABELS = ["(a)", "(b)", "(c)"]
TITLES       = ["Acc. Predicted Hits", "True Hit Ratio", "Uncertainty | Distances"]
Y_LABELS     = ["Acc. Predicted Hits", "True / total hits", "Mean transformed distance"]

def fmt_kilo(x, _):
    return f"{int(x/1000)}k" if x >= 1000 else ("0" if x == 0 else f"{int(x)}")
def to_run_array(series_list, full_index):
    return np.array([
        s.reindex(full_index).ffill().fillna(0).to_numpy() for s in series_list
    ])

fig, axes = plt.subplots(1, 3, figsize=(9, 3), dpi=600, constrained_layout=True, sharex=True)

for i, ax in enumerate(axes.flatten()):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(direction="out", which="both", length=3, width=0.8)
    ax.grid(True, alpha=0.18, linestyle=":", linewidth=0.6, color="#888888")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_kilo))
    ax.set_title(TITLES[i], fontweight="bold", pad=20)
    ax.set_xlabel("RL step")
    ax.set_ylabel(Y_LABELS[i])
    # Panel label (a), (b), (c) in upper-left corner
    ax.text(-0.15, 1.4, PANEL_LABELS[i], transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top", ha="left")

#Data processing & plotting (logic unchanged) 
for file, col in zip(mid, COLORS):
    files = glob(file)
    print(files)

    accumulated_hits_list      = []
    accumulated_true_hits_list = []
    exist = False

    for exp in files:
        if os.path.exists(exp):
            exist = True
            df = pd.read_csv(exp)
            df = df.dropna(subset=["Scaffold"])
            df = df.round(4).dropna()
            if df["step"].max() > 8000:
                print(df["step"].max())
                mask = (
                (df["step"]<10000) &
                (df["UncertaintyLogP (raw)"] > 2.05) &
                (df["UncertaintyLogP (raw)"] < 2.95) &
                (df["UncertaintyBertz (raw)"] > 810) &
                (df["UncertaintyBertz (raw)"] < 990) &
                (df["MW"] > 0.65) &
                (df["TPSA"] > 0.65)
                )
                hits        = df[mask].copy()
                unique_hits = hits.drop_duplicates(subset="Scaffold").reset_index(drop=True)
                true_hits   = unique_hits[
                    (unique_hits["logp_actual (UncertaintyLogP)"] > 2.05) &
                    (unique_hits["logp_actual (UncertaintyLogP)"] < 2.95) &
                    (unique_hits["Bertz_actual (UncertaintyBertz)"] > 810) &
                    (unique_hits["Bertz_actual (UncertaintyBertz)"] < 990)
                ]
                accumulated_hits_list.append(unique_hits["step"].value_counts().sort_index().cumsum())
                accumulated_true_hits_list.append(true_hits["step"].value_counts().sort_index().cumsum())

    if exist:
        full_index = np.arange(10000)
        reindex = lambda lst: np.array([
            s.reindex(full_index).ffill().fillna(0).to_numpy() for s in lst
        ])
        hits_arr  = reindex(accumulated_hits_list)
        true_arr  = reindex(accumulated_true_hits_list)
        false_arr = hits_arr - true_arr

        mean_hits = hits_arr.mean(0)
        mean_true = true_arr.mean(0)
        mean_false = false_arr.mean(0)
        std_hits  = hits_arr.std(0)
        std_true  = true_arr.std(0)
        std_false  = false_arr.std(0)

        kw_fill = dict(alpha=0.18, color=col, linewidth=0)
        kw_line = dict(linewidth=1.2, color=col)

        axes[0].plot(full_index, mean_hits, **kw_line)
        axes[0].fill_between(full_index, mean_hits - std_hits, mean_hits + std_hits, **kw_fill)

        if "NoNoise"  not in files[0]:
            
            denom      = np.where(mean_hits[100:] > 0, mean_hits[100:], np.nan)
            ratio_mean = mean_true[100:] / denom
            axes[1].plot(full_index[100:], ratio_mean, **kw_line)
            axes[1].set_ylim(0.25, 1)

for pattern, col in zip(mid, COLORS):
    files = sorted(glob(pattern))
    score_list, logp_list, bertz_list, simEGFR_list, simDRD2_list = [], [], [], [], []

    for exp in files:
        if not os.path.exists(exp):
            continue

        df = pd.read_csv(exp)
        df = df.dropna(subset=["Scaffold"]).round(4).dropna()
        df = df[df["step"] < 10000].copy()
        if df["step"].max() <= 8000:
            continue

        cols = ["DistanceEGFR", "DistanceDRD2"]
        step_means = df.groupby("step")[cols].mean()

        simEGFR_list.append(1-step_means["DistanceEGFR"])
        simDRD2_list.append(1-step_means["DistanceDRD2"])

    full_index = np.arange(10000)
    simEGFR_arr = to_run_array(simEGFR_list, full_index)
    simDRD2_arr = to_run_array(simDRD2_list, full_index)

    simsConj = np.sqrt(simEGFR_arr * simDRD2_arr)
    #scoresConj = (logp_arr + bertz_arr)/2
    #simsConj = (simEGFR_arr + simDRD2_arr)/2
    mean_v = simsConj.mean(axis=0)
    std_v = simsConj.std(axis=0)

    mean_s = gaussian_filter1d(mean_v, sigma=8)
    axes[2].plot(full_index, mean_s, color=col, linewidth=1)
    axes[2].fill_between(
            full_index,
            mean_s - std_v,
            mean_s + std_v,
            color=col,
            alpha=0.18,
            linewidth=0,
        )
    axes[2].set_ylim(0, 1)



# Legend 
handles = [
    Line2D([0], [0], color=col, lw=1.6, label=lbl, solid_capstyle="round")
    for lbl, col in zip(LABELS, COLORS)
]
blank = Line2D([0], [0], color="none", label="")
# Insert blank at index 2 so row2 reads: [blank, item3, item4] → centered
handles_padded = handles[:1] + [blank] + handles[1:]
fig.legend(
    handles=handles_padded,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.2),
    ncol=3,
    frameon=False,
    handlelength=1.6,
    handletextpad=0.5,
    columnspacing=1.2,
    labelspacing=0.35,
)

plt.tight_layout()
plt.savefig("03_results/accHits_Mid_Multi_midS.pdf", dpi=300, bbox_inches="tight")
plt.savefig("03_results/accHits_Mid_Multi_midS.png", dpi=300, bbox_inches="tight")
plt.show()

### ANALYSIS OF CHEMPROP

#### Chemprop point estimation

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import Chem

from glob import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import os
from scipy.ndimage import gaussian_filter1d


# Colorblind-safe palette (Wong 2011)
LABELS = [
    "No Noise",
    "Noisy Component",
    "Noisy Component + Loss Modulation",
    "Noisy Component + Score Modulation",
    "Noisy Component + Score & Loss Modulation",
]


COLORS = [
    "#555555",
    "#70C267",
    "#6D85BE",  
    "#E66532",
    "#9B2F70" 

]



enes = ["./02_RL/chempropUn8/ChempropUnc_w0_SF_w1_N5_NoNoise_1*.csv",
        "./02_RL/chempropUn8/ChempropUnc_w0_SF_w1_N5_1*.csv",
        "./02_RL/chempropUn8/ChempropUnc_w0_SF_w1_N5_Reward_1*.csv",
        "./02_RL/chempropUn8/ChempropUnc_w1_SF_w1_N5_1*.csv",
        "./02_RL/chempropUn8/ChempropUnc_w1_SF_w1_N5_Reward_1*.csv"]


# Figure layout 
PANEL_LABELS = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)", "(h)", "(i)"]
TITLES       = ["Acc. Predicted Hits", "Accumulated False Hits", "True Hit Ratio", "Predictor Score", "Oracle Score", r"$\Delta$Predictor/Oracle", "Uncertainty Predictor", "Uncertainty Oracle", "Similarity EGFR"]
Y_LABELS     = ["Acc. Predicted Hits", "Accumulated false hits", "True / total hits", "Predictor Score", "Oracle Score", r"$\Delta$Predictor/Oracle", "Uncertainty Predictor", "Uncertainty Oracle", "Similarity EGFR"]

def fmt_kilo(x, _):
    return f"{int(x/1000)}k" if x >= 1000 else ("0" if x == 0 else f"{int(x)}")
def to_run_array(series_list, full_index):
    return np.array([
        s.reindex(full_index).ffill().fillna(0).to_numpy() for s in series_list
    ])

fig, axes = plt.subplots(3, 3, figsize=(9, 7.5), dpi=600, constrained_layout=True, sharex=True)

for i, ax in enumerate(axes.flatten()):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(direction="out", which="both", length=3, width=0.8)
    ax.grid(True, alpha=0.18, linestyle=":", linewidth=0.6, color="#888888")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_kilo))
    ax.set_title(TITLES[i], fontweight="bold", pad=20)
    ax.set_xlabel("RL step")
    ax.set_ylabel(Y_LABELS[i])
    # Panel label (a), (b), (c) in upper-left corner
    ax.text(-0.15, 1.3, PANEL_LABELS[i], transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top", ha="left")

# ── Data processing & plotting (logic unchanged) ──────────────────────────────
for file, col in zip(enes, COLORS):
    files = glob(file)
    print(files)

    accumulated_hits_list      = []
    accumulated_true_hits_list = []
    exist = False

    for exp in files:
        if os.path.exists(exp):
            exist = True
            df = pd.read_csv(exp)
            df = df.dropna(subset=["Scaffold"])
            df = df.round(4).dropna()
            if df["step"].max() >= 3999:
                print(df["step"].max())
                mask = (
                    (df["step"]<4000) &
                    (df["Chemprop (raw)"] > 6) &
                    (df["MW"] > 0.65) &
                    (df["TPSA"] > 0.65)
                )
                hits        = df[mask].copy()
                unique_hits = hits.drop_duplicates(subset="Scaffold").reset_index(drop=True)
                true_hits   = unique_hits[
                    (unique_hits["OracleEvaluator (raw)"] > 6)
                ]
                accumulated_hits_list.append(unique_hits["step"].value_counts().sort_index().cumsum())
                accumulated_true_hits_list.append(true_hits["step"].value_counts().sort_index().cumsum())

    if exist:
        full_index = np.arange(min([len(a) for a in accumulated_hits_list]))
        reindex = lambda lst: np.array([
            s.reindex(full_index).ffill().fillna(0).to_numpy() for s in lst
        ])
        hits_arr  = reindex(accumulated_hits_list)
        true_arr  = reindex(accumulated_true_hits_list)
        false_arr = hits_arr - true_arr

        mean_hits = hits_arr.mean(0)
        mean_true = true_arr.mean(0)
        mean_false = false_arr.mean(0)
        std_hits  = hits_arr.std(0)
        std_true  = true_arr.std(0)
        std_false  = false_arr.std(0)

        kw_fill = dict(alpha=0.18, color=col, linewidth=0)
        kw_line = dict(linewidth=1.2, color=col)

        axes[0,0].plot(full_index, mean_hits, **kw_line)
        axes[0,0].fill_between(full_index, mean_hits - std_hits, mean_hits + std_hits, **kw_fill)

        axes[0,1].plot(full_index, mean_false, **kw_line)
        axes[0,1].fill_between(full_index, mean_false - std_false, mean_false + std_false, **kw_fill)
        
        if "NoNoise"  not in files[0]:
            # Guard against division by zero in ratio panel
            denom      = np.where(mean_hits[100:] > 0, mean_hits[100:], np.nan)
            ratio_mean = mean_true[100:] / denom
            axes[0,2].plot(full_index[100:], ratio_mean, **kw_line)
            axes[0,2].set_ylim(0.2, 1)

for pattern, col in zip(enes, COLORS):
    files = sorted(glob(pattern))
    predictor_list, oracle_list, differences_list, uncertainty_predictor_list, uncertainty_oracle_list, sim_list = [], [], [], [], [], []

    for exp in files:
        if not os.path.exists(exp):
            continue

        df = pd.read_csv(exp)
        df = df.dropna(subset=["Scaffold"]).round(4).dropna()
        df = df[df["step"] < 4000].copy()
        if df["step"].max() <= 3998:
            continue
        df["differenceChemprop"] = abs(df["OracleEvaluator (raw)"] - df["Chemprop (raw)"])
        cols = ["DistanceEGFR", "Chemprop (raw)","OracleEvaluator","OracleEvaluator (raw)","uncertainty","uncertainty (raw)","differenceChemprop","uncertainties (OracleEvaluator)","uncertainties (Chemprop)"]
        step_means = df.groupby("step")[cols].mean()

        predictor_list.append(step_means["Chemprop (raw)"])
        oracle_list.append(step_means["OracleEvaluator (raw)"])
        differences_list.append(step_means["differenceChemprop"])
        uncertainty_predictor_list.append(step_means["uncertainties (Chemprop)"])
        uncertainty_oracle_list.append(step_means["uncertainties (OracleEvaluator)"])
        sim_list.append(step_means["DistanceEGFR"])


    if not predictor_list:
        continue

    full_index =np.arange(min([len(a) for a in predictor_list]))
    predictor_arr = to_run_array(predictor_list, full_index)
    oracle_arr = to_run_array(oracle_list, full_index)
    difference_arr = to_run_array(differences_list, full_index)
    uncertainty_predictor_arr = to_run_array(uncertainty_predictor_list, full_index)
    uncertainty_oracle_arr = to_run_array(uncertainty_oracle_list, full_index)
    sim_arr = to_run_array(sim_list, full_index)

    metric_arrays = [predictor_arr, oracle_arr, difference_arr, uncertainty_predictor_arr, uncertainty_oracle_arr, sim_arr]

    for ax_idx, arr in enumerate(metric_arrays):

        if ax_idx in [0,2,3]: # Predictor Score and Uncertainty Predictor panels
            if "NoNoise" in pattern:
                continue

        row=1
        if ax_idx > 2:
            ax_idx=ax_idx-3
            row=2

        mean_v = arr.mean(axis=0)
        std_v = arr.std(axis=0)

        mean_s = gaussian_filter1d(mean_v, sigma=8)

        #mean_s, std_s = mean_v, std_v

        axes[row,ax_idx].plot(full_index, mean_s, color=col, linewidth=1)
        axes[row,ax_idx].fill_between(
            full_index,
            mean_s - std_v,
            mean_s + std_v,
            color=col,
            alpha=0.18,
            linewidth=0,
        )




# Legend 
handles = [
    Line2D([0], [0], color=col, lw=1.6, label=lbl, solid_capstyle="round")
    for lbl, col in zip(LABELS, COLORS)
]
blank = Line2D([0], [0], color="none", label="")
# Insert blank at index 2 so row2 reads: [blank, item3, item4] → centered
handles_padded = handles[:1] + [blank] + handles[1:]
fig.legend(
    handles=handles_padded,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.1),
    ncol=3,
    frameon=False,
    handlelength=1.6,
    handletextpad=0.5,
    columnspacing=1.2,
    labelspacing=0.35,
)

plt.tight_layout()
plt.savefig("03_results/chemprop.pdf", dpi=300, bbox_inches="tight")
plt.savefig("03_results/chemprop.png", dpi=300, bbox_inches="tight")
plt.show()

#### Chemprop average result

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import Chem

from glob import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import os
from scipy.ndimage import gaussian_filter1d



LABELS = [
    "Point Estimate",
    "Probabilistic Scoring Function"
]

##Depending which plot to make comment or uncomment the corresponding color and data path sections below.



#COLORS = ["#1B5E49",
#          "#70C267",]
#COLORS = ["#0036B4",
#    "#6D85BE",]
#COLORS = ["#872600",
#    "#E66532"]
COLORS = ["#32001E",
     "#9B2F70"]



# ── Data paths (unchanged) ────────────────────────────────────────────────────

#enes = ["./02_RL/chempropUn8/ChempropUnc_w0_SF_w1_N5_1*.csv",
#        "./02_RL/chempropUn8Av/ChempropUnc_w0_SF_w1_N5Av_1*.csv"]
#enes = ["./02_RL/chempropUn8/ChempropUnc_w0_SF_w1_N5_Reward_1*.csv",
#        "./02_RL/chempropUn8Av/ChempropUnc_w0_SF_w1_N5_RewardAv_1*.csv"]
#enes = ["./02_RL/chempropUn8/ChempropUnc_w1_SF_w1_N5_1*.csv",
#        "./02_RL/chempropUn8Av/ChempropUnc_w1_SF_w1_N5Av_1*.csv"]
enes = ["./02_RL/chempropUn8/ChempropUnc_w1_SF_w1_N5_Reward_1*.csv",
        "./02_RL/chempropUn8Av/ChempropUnc_w1_SF_w1_N5_RewardAv_1*.csv"]

# ── Figure layout ─────────────────────────────────────────────────────────────
PANEL_LABELS = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)", "(h)", "(i)"]
TITLES       = ["Acc. Predicted Hits", "Accumulated False Hits", "True Hit Ratio", "Predictor Score", "Oracle Score", r"$\Delta$ Predictor/Oracle", "Uncertainty Predictor", "Uncertainty Oracle", "Similarity EGFR"]
Y_LABELS     = ["Acc. Predicted Hits", "Accumulated false hits", "True / total hits", "Predictor Score", "Oracle Score", r"$\Delta$ Chemprop vs Oracle", "Uncertainty Predictor", "Uncertainty Oracle", "Similarity EGFR"]

def fmt_kilo(x, _):
    return f"{int(x/1000)}k" if x >= 1000 else ("0" if x == 0 else f"{int(x)}")
def to_run_array(series_list, full_index):
    return np.array([
        s.reindex(full_index).ffill().fillna(0).to_numpy() for s in series_list
    ])

fig, axes = plt.subplots(3, 3, figsize=(9, 7.5), dpi=600, constrained_layout=True, sharex=True)

for i, ax in enumerate(axes.flatten()):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(direction="out", which="both", length=3, width=0.8)
    ax.grid(True, alpha=0.18, linestyle=":", linewidth=0.6, color="#888888")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_kilo))
    ax.set_title(TITLES[i], fontweight="bold", pad=20)
    ax.set_xlabel("RL step")
    ax.set_ylabel(Y_LABELS[i])
    # Panel label (a), (b), (c) in upper-left corner
    ax.text(-0.15, 1.3, PANEL_LABELS[i], transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top", ha="left")

# ── Data processing & plotting (logic unchanged) ──────────────────────────────
for file, col in zip(enes, COLORS):
    files = glob(file)
    print(files)

    accumulated_hits_list      = []
    accumulated_true_hits_list = []
    exist = False

    for exp in files:
        if os.path.exists(exp):
            exist = True
            df = pd.read_csv(exp)
            df = df.dropna(subset=["Scaffold"])
            df = df.round(4).dropna()
            if df["step"].max() >= 3988:
                print(df["step"].max())
                mask = (
                    (df["step"]<4000) &
                    (df["Chemprop (raw)"] > 6) &
                    (df["MW"] > 0.65) &
                    (df["TPSA"] > 0.65)
                )
                hits        = df[mask].copy()
                unique_hits = hits.drop_duplicates(subset="Scaffold").reset_index(drop=True)
                true_hits   = unique_hits[
                    (unique_hits["OracleEvaluator (raw)"] > 6)
                ]
                accumulated_hits_list.append(unique_hits["step"].value_counts().sort_index().cumsum())
                accumulated_true_hits_list.append(true_hits["step"].value_counts().sort_index().cumsum())

    if exist:
        full_index = np.arange(min([len(a) for a in accumulated_hits_list]))
        reindex = lambda lst: np.array([
            s.reindex(full_index).ffill().fillna(0).to_numpy() for s in lst
        ])
        hits_arr  = reindex(accumulated_hits_list)
        true_arr  = reindex(accumulated_true_hits_list)
        false_arr = hits_arr - true_arr

        mean_hits = hits_arr.mean(0)
        mean_true = true_arr.mean(0)
        mean_false = false_arr.mean(0)
        std_hits  = hits_arr.std(0)
        std_true  = true_arr.std(0)
        std_false  = false_arr.std(0)

        kw_fill = dict(alpha=0.18, color=col, linewidth=0)
        kw_line = dict(linewidth=1.2, color=col)

        axes[0,0].plot(full_index, mean_hits, **kw_line)
        axes[0,0].fill_between(full_index, mean_hits - std_hits, mean_hits + std_hits, **kw_fill)

        axes[0,1].plot(full_index, mean_false, **kw_line)
        axes[0,1].fill_between(full_index, mean_false - std_false, mean_false + std_false, **kw_fill)

        # Guard against division by zero in ratio panel
        denom      = np.where(mean_hits[100:] > 0, mean_hits[100:], np.nan)
        ratio_mean = mean_true[100:] / denom
        axes[0,2].plot(full_index[100:], ratio_mean, **kw_line)
        axes[0,2].set_ylim(0, 1)

for pattern, col in zip(enes, COLORS):
    files = sorted(glob(pattern))
    predictor_list, oracle_list, differences_list, uncertainty_predictor_list, uncertainty_oracle_list, sim_list = [], [], [], [], [], []

    for exp in files:
        if not os.path.exists(exp):
            continue

        df = pd.read_csv(exp)
        df = df.dropna(subset=["Scaffold"]).round(4).dropna()
        df = df[df["step"] < 4000].copy()
        if df["step"].max() < 3998:
            continue
        df["differenceChemprop"] = abs(df["OracleEvaluator (raw)"] - df["Chemprop (raw)"])
        cols = ["DistanceEGFR", "Chemprop (raw)","OracleEvaluator","OracleEvaluator (raw)","uncertainty","uncertainty (raw)","differenceChemprop","uncertainties (OracleEvaluator)","uncertainties (Chemprop)"]
        step_means = df.groupby("step")[cols].mean()

        predictor_list.append(step_means["Chemprop (raw)"])
        oracle_list.append(step_means["OracleEvaluator (raw)"])
        differences_list.append(abs(step_means["Chemprop (raw)"]-step_means["OracleEvaluator (raw)"]))
        uncertainty_predictor_list.append(step_means["uncertainties (Chemprop)"])
        uncertainty_oracle_list.append(step_means["uncertainties (OracleEvaluator)"])
        sim_list.append(step_means["DistanceEGFR"])


    if not predictor_list:
        continue

    full_index =np.arange(min([len(a) for a in predictor_list]))
    predictor_arr = to_run_array(predictor_list, full_index)
    oracle_arr = to_run_array(oracle_list, full_index)
    difference_arr = to_run_array(differences_list, full_index)
    uncertainty_predictor_arr = to_run_array(uncertainty_predictor_list, full_index)
    uncertainty_oracle_arr = to_run_array(uncertainty_oracle_list, full_index)
    sim_arr = to_run_array(sim_list, full_index)

    metric_arrays = [predictor_arr, oracle_arr, difference_arr, uncertainty_predictor_arr, uncertainty_oracle_arr, sim_arr]

    for ax_idx, arr in enumerate(metric_arrays):

        if ax_idx in [0, 3]: # Predictor Score and Uncertainty Predictor panels
            if "NoNoise" in pattern:
                continue

        row=1
        if ax_idx > 2:
            ax_idx=ax_idx-3
            row=2

        mean_v = arr.mean(axis=0)
        std_v = arr.std(axis=0)

        mean_s = gaussian_filter1d(mean_v, sigma=8)


        axes[row,ax_idx].plot(full_index, mean_s, color=col, linewidth=1)
        axes[row,ax_idx].fill_between(
            full_index,
            mean_s - std_v,
            mean_s + std_v,
            color=col,
            alpha=0.18,
            linewidth=0,
        )




# ── Legend ─────────────────────────────────────────────────────────────────────
handles = [
    Line2D([0], [0], color=col, lw=1.6, label=lbl, solid_capstyle="round")
    for lbl, col in zip(LABELS, COLORS)
]
fig.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5,-0.1),
    ncol=5,
    frameon=False,
    handlelength=1.6,
    handletextpad=0.5,
    columnspacing=1.2,
    labelspacing=0.35,
)

plt.savefig("03_results/chempropAvBoth.pdf", dpi=300, bbox_inches="tight")
plt.savefig("03_results/chempropAvBoth.png", dpi=300, bbox_inches="tight")
plt.show()

### ANALYSIS OF CONFORMAL PREDICTION

In [ ]:

LABELS = [
    "Noisy Component",
    "Noisy Component + Loss Modulation",
    "Noisy Component + Score Modulation",
    "Noisy Component + Score & Loss Modulation",
]


COLORS = [
    "#70C267",
    "#6D85BE",  
    "#E66532",
    "#9B2F70" 
]


# ── Data paths (unchanged) ────────────────────────────────────────────────────

enes=["./02_RL/conformal/DISTw0_SFw1_N5Euc_1*.csv",
      "./02_RL/conformal/DISTw0_SFw1_N5EucReward_1*.csv",
      "./02_RL/conformal/DISTw1_SFw1_N5Euc_1*.csv",
      "./02_RL/conformal/DISTw1_SFw1_N5EucGReward_1*.csv"]



# ── Figure layout ─────────────────────────────────────────────────────────────
PANEL_LABELS = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]
TITLES       = ["Acc. Predicted Hits", "Accumulated False Hits", "True Hit Ratio", "Activity Score", "Uncertainty (Conformal)", "Similarity vs Reference Dataset"]
Y_LABELS     = ["Acc. Predicted Hits", "Accumulated false hits", "True / total hits", "Score (p_active)", "Uncertainty (Conformal)", "Mean similarity"]

def fmt_kilo(x, _):
    return f"{int(x/1000)}k" if x >= 1000 else ("0" if x == 0 else f"{int(x)}")
def to_run_array(series_list, full_index):
    return np.array([
        s.reindex(full_index).ffill().fillna(0).to_numpy() for s in series_list
    ])

fig, axes = plt.subplots(2, 3, figsize=(9, 5), dpi=600, constrained_layout=True, sharex=True)

for i, ax in enumerate(axes.flatten()):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(direction="out", which="both", length=3, width=0.8)
    ax.grid(True, alpha=0.18, linestyle=":", linewidth=0.6, color="#888888")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_kilo))
    ax.set_title(TITLES[i], fontweight="bold", pad=20)
    ax.set_xlabel("RL step")
    ax.set_ylabel(Y_LABELS[i])
    # Panel label (a), (b), (c) in upper-left corner
    ax.text(-0.15, 1.3, PANEL_LABELS[i], transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top", ha="left")

# ── Data processing & plotting (logic unchanged) ──────────────────────────────
for file, col in zip(enes, COLORS):
    files = glob(file)
    print(files)

    accumulated_hits_list      = []
    accumulated_true_hits_list = []
    exist = False
    significance=0.05
    for exp in files:
        if os.path.exists(exp):
            exist = True
            df = pd.read_csv(exp)
            df = df.dropna(subset=["Scaffold"])
            df = df.round(4).dropna()
            if df["step"].max() > 500:
                print(max(df["step"]))
                mask = (
                    (df["step"] < 10000) &
                    (df["MW"] > 0.65) &
                    (df["TPSA"] > 0.65)
                )
                hits        = df[mask].copy()
                unique_hits = hits.drop_duplicates(subset="Scaffold").reset_index(drop=True)
                true_hits   = unique_hits[
                    (unique_hits["p_inactive (unc_CP)"] <= significance)&
                    (unique_hits["p_active (unc_CP)"] > significance)
                ]
                accumulated_hits_list.append(unique_hits["step"].value_counts().sort_index().cumsum())
                accumulated_true_hits_list.append(true_hits["step"].value_counts().sort_index().cumsum())

    if exist:
        full_index = np.arange(10000)
        reindex = lambda lst: np.array([
            s.reindex(full_index).ffill().fillna(0).to_numpy() for s in lst
        ])
        hits_arr  = reindex(accumulated_hits_list)
        true_arr  = reindex(accumulated_true_hits_list)
        false_arr = hits_arr - true_arr

        mean_hits = hits_arr.mean(0)
        mean_true = true_arr.mean(0)
        mean_false = false_arr.mean(0)
        std_hits  = hits_arr.std(0)
        std_true  = true_arr.std(0)
        std_false  = false_arr.std(0)

        kw_fill = dict(alpha=0.18, color=col, linewidth=0)
        kw_line = dict(linewidth=1.2, color=col)

        axes[0,0].plot(full_index, mean_hits, **kw_line)
        axes[0,0].fill_between(full_index, mean_hits - std_hits, mean_hits + std_hits, **kw_fill)

        axes[0,1].plot(full_index, mean_false, **kw_line)
        axes[0,1].fill_between(full_index, mean_false - std_false, mean_false + std_false, **kw_fill)

        # Guard against division by zero in ratio panel
        denom      = np.where(mean_hits[100:] > 0, mean_hits[100:], np.nan)
        ratio_mean = mean_true[100:] / denom
        axes[0,2].plot(full_index[100:], ratio_mean, **kw_line)
        axes[0,2].set_ylim(0, 1)

for pattern, col in zip(enes, COLORS):
    files = sorted(glob(pattern))
    score_list, logp_list, sim_list = [], [], []

    for exp in files:
        if not os.path.exists(exp):
            continue

        df = pd.read_csv(exp)
        df = df.dropna(subset=["Scaffold"]).round(4).dropna()
        df = df[df["step"] < 10000].copy()
        if df["step"].max() <= 500:
            continue

        cols = ["conformalPrediction", "unc_CP", "DistanceEGFR"]
        step_means = df.groupby("step")[cols].mean()

        score_list.append(step_means["conformalPrediction"])
        logp_list.append(1-step_means["unc_CP"])
        sim_list.append(step_means["DistanceEGFR"])

    if not score_list:
        continue
    
    full_index = np.arange(10000)
    score_arr = to_run_array(score_list, full_index)
    logp_arr = to_run_array(logp_list, full_index)
    sim_arr = to_run_array(sim_list, full_index)

    metric_arrays = [score_arr, logp_arr, sim_arr]

    for ax_idx, arr in enumerate(metric_arrays):
        mean_v = arr.mean(axis=0)
        std_v = arr.std(axis=0)

        mean_s = gaussian_filter1d(mean_v, sigma=8)


        axes[1,ax_idx].plot(full_index, mean_s, color=col, linewidth=1)
        axes[1,ax_idx].fill_between(
            full_index,
            mean_s - std_v,
            mean_s + std_v,
            color=col,
            alpha=0.18,
            linewidth=0,
        )
        #axes[1,ax_idx].set_ylim(0, 1)



# Legend 
handles = [
    Line2D([0], [0], color=col, lw=1.6, label=lbl, solid_capstyle="round")
    for lbl, col in zip(LABELS, COLORS)
]
blank = Line2D([0], [0], color="none", label="")
# Insert blank at index 2 so row2 reads: [blank, item3, item4] → centered
handles_padded = handles[:1] + [blank] + handles[1:]
fig.legend(
    handles=handles_padded,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.1),
    ncol=3,
    frameon=False,
    handlelength=1.6,
    handletextpad=0.5,
    columnspacing=1.2,
    labelspacing=0.35,
)

plt.tight_layout()
plt.savefig("03_results/conformal.pdf", dpi=300, bbox_inches="tight")
plt.savefig("03_results/conformal.png", dpi=300, bbox_inches="tight")
plt.show()